# CSR Crawler
Crawl web dong (Playwright)

In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')
WORK_DIR = '/content/drive/MyDrive/Crawl_Data/CrawlData'
DOWNLOAD_DIR = os.path.join(WORK_DIR, 'downloaded_files2')
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print('Work:', WORK_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Work: /content/drive/MyDrive/Crawl_Data/CrawlData


In [3]:
!pip install playwright beautifulsoup4 lxml aiohttp nest_asyncio -q
!playwright install chromium
!playwright install-deps

Installing dependencies...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 3,917 B in 1s (2,866 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
import asyncio, re, json, aiohttp, nest_asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, unquote
from datetime import datetime
nest_asyncio.apply()

INPUT_FILE = os.path.join(WORK_DIR, 'csr_urls.json')
OUTPUT_FILE = os.path.join(WORK_DIR, 'output_csr.json')
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

In [5]:
def get_filename(url):
    path = urlparse(url).path
    filename = unquote(os.path.basename(path))
    if not filename or filename == '/': filename = f'file_{hash(url) % 100000}'
    if '.' not in filename: filename = filename + '.bin'
    return filename

def is_downloadable(url, exts): return any(urlparse(url.lower()).path.endswith(e) for e in exts)
def is_same_domain(url, base): return urlparse(base).netloc == urlparse(url).netloc

def extract_links(html, base, selector=None, exts=None, require_ext=False):
    soup = BeautifulSoup(html, 'lxml')
    links, seen = [], set()
    if selector and selector.get('selector') and selector.get('type') == 'css':
        for elem in soup.select(selector['selector']):
            if elem.name == 'a' and elem.get('href'):
                url = urljoin(base, elem['href'])
                if url not in seen: seen.add(url); links.append(url)
            for a in elem.find_all('a', href=True):
                url = urljoin(base, a['href'])
                if url not in seen: seen.add(url); links.append(url)
    else:
        for a in soup.find_all('a', href=True):
            url = urljoin(base, a['href'])
            if url not in seen:
                if require_ext and exts:
                    if is_downloadable(url, exts): seen.add(url); links.append(url)
                else: seen.add(url); links.append(url)
    return links

In [6]:
async def download(session, url, folder, fname):
    r = {'url': url, 'filename': fname, 'status': 'pending'}
    try:
        # Kiểm tra file đã tồn tại
        path = os.path.join(folder, fname)
        if os.path.exists(path):
            r['status'] = 'skipped'
            r['reason'] = 'already_exists'
            r['path'] = path
            r['size'] = os.path.getsize(path)
            return r

        async with session.get(url, timeout=aiohttp.ClientTimeout(120)) as resp:
            if resp.status == 200:
                if 'Content-Disposition' in resp.headers:
                    cd = resp.headers['Content-Disposition']
                    for p in [r"filename\*=UTF-8''(.+)", r'filename="(.+)"', r"filename='(.+)'", r'filename=([^;\s]+)']:
                        m = re.search(p, cd)
                        if m: r['filename'] = unquote(m.group(1).strip()); break

                # Kiểm tra lại với filename mới từ Content-Disposition
                path = os.path.join(folder, r['filename'])
                if os.path.exists(path):
                    r['status'] = 'skipped'
                    r['reason'] = 'already_exists'
                    r['path'] = path
                    r['size'] = os.path.getsize(path)
                    return r

                c = 1; b, ext = os.path.splitext(path)
                while os.path.exists(path): path = f'{b}_{c}{ext}'; c += 1

                # CHUNK DOWNLOAD IMPLEMENTATION
                with open(path, 'wb') as f:
                    async for chunk in resp.content.iter_chunked(1024*1024): # 1MB chunks
                        f.write(chunk)

                r['status'], r['size'], r['path'] = 'success', os.path.getsize(path), path
            else: r['status'], r['error'] = 'failed', f'HTTP {resp.status}'
    except Exception as e: r['status'], r['error'] = 'failed', str(e)
    return r

In [7]:
async def crawl_multi_level(cfg, options):
    url = cfg.get('url')
    levels = cfg.get('levels', [])
    exts = cfg.get('file_extensions', ['.pdf'])
    delay = options.get('delay_between_requests', 1)
    max_files = options.get('max_files', 0)
    result = {'url': url, 'levels': [], 'files': [], 'status': 'pending'}

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        try:
            print(f'\n  Accessing: {url}')
            await page.goto(url, wait_until='networkidle', timeout=60000)
            await page.wait_for_timeout(2000)

            # Nhấn nút "Tìm kiếm"
            try:
                search_btn = page.locator('#searchSubmit')
                if await search_btn.count() > 0:
                    print('  Clicking search button...')
                    await search_btn.click()
                    await page.wait_for_timeout(3000)
            except Exception as e:
                print(f'  Search button click failed: {str(e)[:40]}')

            page_num = 1
            max_pagination = 100

            while page_num <= max_pagination:
                print(f'\n  Page {page_num}:')
                await page.wait_for_timeout(2000)

                # Tìm tất cả nút "Tải về"
                download_btns = page.locator('a[href*="ShowDialogDownload"]')
                btn_count = await download_btns.count()
                print(f'    Found {btn_count} download buttons')

                for i in range(btn_count):
                    if max_files > 0 and len(result['files']) >= max_files:
                        print(f'    Reached max_files limit ({max_files})')
                        break

                    try:
                        # Click nút "Tải về" thứ i
                        await download_btns.nth(i).click()
                        await page.wait_for_timeout(1500)

                        # Tìm link download trong popup
                        download_links = page.locator('a[href*="downloadfile"]')
                        link_count = await download_links.count()

                        if link_count > 0:
                            # Lấy href và filename
                            href = await download_links.first.get_attribute('href')
                            # Parse: javascript:downloadfile('filename.doc','/path/to/file.doc')
                            match = re.search(r"downloadfile\('([^']+)','([^']+)'\)", href)
                            if match:
                                filename = match.group(1)
                                file_path = match.group(2)
                                file_url = urljoin(url, file_path)

                                print(f'    [{i+1}] Downloading: {filename[:40]}')

                                # Download file
                                cookies_list = await page.context.cookies()
                                cookies = {c['name']: c['value'] for c in cookies_list}

                                async with aiohttp.ClientSession(headers=HEADERS, cookies=cookies) as s:
                                    dl_result = await download(s, file_url, DOWNLOAD_DIR, filename)
                                    result['files'].append(dl_result)

                                    status = 'OK' if dl_result['status'] == 'success' else ('SKIP' if dl_result['status'] == 'skipped' else 'FAIL')
                                    print(f'        [{status}] {filename[:40]}')

                        # Đóng popup (nếu có)
                        try:
                            close_btn = page.locator('a.close, button.close, [onclick*="close"]')
                            if await close_btn.count() > 0:
                                await close_btn.first.click()
                                await page.wait_for_timeout(500)
                        except:
                            pass

                    except Exception as e:
                        print(f'    Error at button {i+1}: {str(e)[:40]}')

                if max_files > 0 and len(result['files']) >= max_files:
                    break

                # Chuyển trang
                try:
                    next_page_num = page_num + 1
                    next_link = page.locator(f'a[href*="LoadPage({next_page_num})"]')

                    if await next_link.count() > 0:
                        print(f'    Going to page {next_page_num}...')
                        await next_link.click()
                        await page.wait_for_timeout(3000)
                        page_num += 1
                    else:
                        print('    No more pages')
                        break
                except Exception as e:
                    print(f'    Pagination error: {str(e)[:40]}')
                    break

            result['levels'].append({
                'name': 'Download files',
                'pages_crawled': page_num,
                'total_downloaded': len(result['files'])
            })
            result['status'] = 'success'

        except Exception as e:
            result['status'], result['error'] = 'failed', str(e)
        finally:
            await browser.close()

    return result

In [8]:
async def main():
    with open(INPUT_FILE, 'r', encoding='utf-8') as f: data = json.load(f)
    urls, opts = data.get('urls', []), data.get('options', {})
    print(f'URLs: {len(urls)}')
    if opts.get('max_files'): print(f'Max files: {opts["max_files"]}')
    print('='*50)
    results = []
    for i, cfg in enumerate(urls, 1):
        url, mode = cfg.get('url'), cfg.get('crawl_mode', 'one_level')
        print(f'\n[{i}] {url[:50]}...')
        print(f'  Mode: {mode}')
        if mode == 'multi_level': r = await crawl_multi_level(cfg, opts)
        elif mode == 'two_level': r = await crawl_two_level(cfg, opts)
        else: r = await crawl_one_level(cfg, opts)
        ok = sum(1 for f in r['files'] if f['status']=='success')
        print(f'\n  Downloaded: {ok}/{len(r["files"])}')
        results.append(r)
    return results

all_results = asyncio.get_event_loop().run_until_complete(main())
print('\nDone!')

URLs: 1

[1] https://www.mod.gov.vn/home/news...
  Mode: news


NameError: name 'crawl_one_level' is not defined

In [ ]:
output = {'results': all_results, 'summary': {'urls': len(all_results), 'total_files': sum(len(r['files']) for r in all_results), 'downloaded': sum(sum(1 for f in r['files'] if f['status']=='success') for r in all_results)}, 'crawled_at': datetime.now().isoformat()}
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f: json.dump(output, f, indent=2, ensure_ascii=False)
print(f'Total: {output["summary"]["total_files"]}')
print(f'Downloaded: {output["summary"]["downloaded"]}')

In [9]:

import asyncio, re, json, nest_asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime
nest_asyncio.apply()

INPUT_FILE = os.path.join(WORK_DIR, 'csr_urls.json')
OUTPUT_FILE = os.path.join(WORK_DIR, 'output_news.json')

def sanitize_filename(title):
    filename = re.sub(r'[^\w\s-]', '', title)
    filename = re.sub(r'[-\s]+', '_', filename)
    return filename[:100]

async def crawl_news_article(page, url, base_url):
    result = {'url': url, 'status': 'pending'}
    try:
        full_url = urljoin(base_url, url)
        print(f'      Opening: {url[:60]}...')

        await page.goto(full_url, wait_until='networkidle', timeout=60000)
        await page.wait_for_timeout(2000)

        title_elem = page.locator('h1.titleHotNews span')
        if await title_elem.count() > 0:
            result['title'] = await title_elem.first.inner_text()
        else:
            result['title'] = 'No title'

        content_elem = page.locator('div.contentDetail')
        if await content_elem.count() > 0:
            result['content'] = await content_elem.first.inner_text()
            result['content_html'] = await content_elem.first.inner_html()
        else:
            result['content'] = ''
            result['content_html'] = ''

        author_elem = page.locator('p.author.text-right')
        if await author_elem.count() > 0:
            result['author'] = await author_elem.first.inner_text()
        else:
            result['author'] = ''

        result['status'] = 'success'

        filename = sanitize_filename(result['title'])
        txt_path = os.path.join(NEWS_DIR, f'{filename}.txt')

        counter = 1
        while os.path.exists(txt_path):
            txt_path = os.path.join(NEWS_DIR, f'{filename}_{counter}.txt')
            counter += 1

        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(f"TIÊU ĐỀ: {result['title']}\n")
            f.write(f"URL: {full_url}\n")
            f.write(f"TÁC GIẢ: {result['author']}\n")
            f.write(f"\n{'='*80}\n\n")
            f.write(result['content'])

        result['saved_file'] = txt_path
        print(f'      [OK] Saved: {filename[:40]}.txt')

    except Exception as e:
        result['status'] = 'failed'
        result['error'] = str(e)
        print(f'      [FAIL] {str(e)[:50]}')

    return result

async def crawl_news_list(cfg, options):
    url = cfg.get('url')
    max_pages = options.get('max_pages', 100)
    max_articles = options.get('max_files', 0)
    delay = options.get('delay_between_requests', 2)

    result = {'url': url, 'articles': [], 'status': 'pending'}

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        try:
            print(f'\n  Accessing: {url}')
            await page.goto(url, wait_until='networkidle', timeout=60000)
            await page.wait_for_timeout(3000)

            # Scroll để load nội dung
            await page.evaluate('window.scrollTo(0, document.body.scrollHeight)')
            await page.wait_for_timeout(2000)
            await page.evaluate('window.scrollTo(0, 0)')
            await page.wait_for_timeout(1000)

            current_page = 1

            while current_page <= max_pages:
                print(f'\n  === Page {current_page} ===')
                await page.wait_for_timeout(delay * 1000)

                # Lấy HTML và parse với BeautifulSoup
                html = await page.content()
                soup = BeautifulSoup(html, 'lxml')

                # Tìm tất cả bài viết (kể cả ẩn)
                article_items = soup.select('div.itemnew-inlist')
                item_count = len(article_items)
                print(f'    Found {item_count} articles')

                if item_count == 0:
                    print('    WARNING: No articles found! Check HTML structure.')
                    break

                for i, item in enumerate(article_items):
                    if max_articles > 0 and len(result['articles']) >= max_articles:
                        print(f'    Reached max_articles limit ({max_articles})')
                        break

                    try:
                        # Lấy link bài viết
                        link_tag = item.select_one('h5.titleLeftNews a')
                        if link_tag and link_tag.get('href'):
                            article_url = link_tag['href']
                            article_title = link_tag.get_text(strip=True)

                            # Lấy thời gian
                            time_tag = item.select_one('div.time-post')
                            article_time = time_tag.get_text(strip=True) if time_tag else ''

                            print(f'    [{i+1}] {article_title[:50]}...')
                            print(f'        Time: {article_time}')

                            # Crawl chi tiết
                            article_data = await crawl_news_article(page, article_url, url)
                            article_data['time_post'] = article_time
                            article_data['title_from_list'] = article_title

                            result['articles'].append(article_data)

                            # Quay lại trang danh sách
                            await page.goto(url if current_page == 1 else page.url, wait_until='networkidle')
                            await page.wait_for_timeout(delay * 1000)

                    except Exception as e:
                        print(f'    Error at article {i+1}: {str(e)[:50]}')

                if max_articles > 0 and len(result['articles']) >= max_articles:
                    break

                # Chuyển trang
                try:
                    next_page_num = current_page + 1
                    next_link = page.locator(f'div.page a[title*="page {next_page_num}"]')

                    if await next_link.count() > 0:
                        print(f'    Going to page {next_page_num}...')
                        await next_link.first.click()
                        await page.wait_for_timeout(3000)

                        # Scroll lại
                        await page.evaluate('window.scrollTo(0, document.body.scrollHeight)')
                        await page.wait_for_timeout(2000)

                        current_page += 1
                    else:
                        print('    No more pages')
                        break

                except Exception as e:
                    print(f'    Pagination error: {str(e)[:50]}')
                    break

            result['status'] = 'success'
            result['total_pages'] = current_page

        except Exception as e:
            result['status'] = 'failed'
            result['error'] = str(e)
        finally:
            await browser.close()

    return result

async def main():
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)

    urls = data.get('urls', [])
    opts = data.get('options', {})

    print(f'URLs: {len(urls)}')
    if opts.get('max_files'):
        print(f'Max articles: {opts["max_files"]}')
    if opts.get('max_pages'):
        print(f'Max pages: {opts["max_pages"]}')
    print('='*50)

    results = []
    for i, cfg in enumerate(urls, 1):
        url = cfg.get('url')
        print(f'\n[{i}] {url}')

        r = await crawl_news_list(cfg, opts)

        success_count = sum(1 for a in r['articles'] if a['status'] == 'success')
        print(f'\n  Crawled: {success_count}/{len(r["articles"])} articles')

        results.append(r)

    return results

all_results = asyncio.get_event_loop().run_until_complete(main())

print('\n' + '='*50)
print('DONE!')
print('='*50)

output = {
    'results': all_results,
    'summary': {
        'urls': len(all_results),
        'total_articles': sum(len(r['articles']) for r in all_results),
        'successful': sum(sum(1 for a in r['articles'] if a['status']=='success') for r in all_results)
    },
    'crawled_at': datetime.now().isoformat()
}

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f'\nTotal articles: {output["summary"]["total_articles"]}')
print(f'Successful: {output["summary"]["successful"]}')
print(f'\nNews saved in: {NEWS_DIR}')
print(f'JSON saved: {OUTPUT_FILE}')

URLs: 1
Max pages: 5

[1] https://www.mod.gov.vn/home/news

  Accessing: https://www.mod.gov.vn/home/news

  === Page 1 ===
    Found 0 articles

  Crawled: 0/0 articles

DONE!

Total articles: 0
Successful: 0

News saved in: /content/drive/MyDrive/Crawl_Data/CrawlData/news_articles
JSON saved: /content/drive/MyDrive/Crawl_Data/CrawlData/output_news.json


In [ ]:
# Version with Playwright Stealth
from google.colab import drive
import os
drive.mount('/content/drive')
WORK_DIR = '/content/drive/MyDrive/Crawl_Data/CrawlData'
NEWS_DIR = os.path.join(WORK_DIR, 'news_articles')
os.makedirs(NEWS_DIR, exist_ok=True)
os.chdir(WORK_DIR)

!pip install playwright-stealth playwright beautifulsoup4 lxml nest_asyncio -q
!playwright install chromium
!playwright install-deps

import asyncio, re, json, nest_asyncio
from playwright.async_api import async_playwright
from playwright_stealth import stealth_async
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime
nest_asyncio.apply()

INPUT_FILE = os.path.join(WORK_DIR, 'csr_urls.json')
OUTPUT_FILE = os.path.join(WORK_DIR, 'output_news.json')

def sanitize_filename(title):
    filename = re.sub(r'[^\w\s-]', '', title)
    filename = re.sub(r'[-\s]+', '_', filename)
    return filename[:100]

async def crawl_news_article(page, url, base_url):
    result = {'url': url, 'status': 'pending'}
    try:
        full_url = urljoin(base_url, url)
        print(f'      Opening: {url[:60]}...')

        await page.goto(full_url, wait_until='domcontentloaded', timeout=60000)
        await page.wait_for_timeout(3000)

        title_elem = page.locator('h1.titleHotNews span')
        if await title_elem.count() > 0:
            result['title'] = await title_elem.first.inner_text()
        else:
            result['title'] = 'No title'

        content_elem = page.locator('div.contentDetail')
        if await content_elem.count() > 0:
            result['content'] = await content_elem.first.inner_text()
            result['content_html'] = await content_elem.first.inner_html()
        else:
            result['content'] = ''
            result['content_html'] = ''

        author_elem = page.locator('p.author.text-right')
        if await author_elem.count() > 0:
            result['author'] = await author_elem.first.inner_text()
        else:
            result['author'] = ''

        result['status'] = 'success'

        filename = sanitize_filename(result['title'])
        txt_path = os.path.join(NEWS_DIR, f'{filename}.txt')

        counter = 1
        while os.path.exists(txt_path):
            txt_path = os.path.join(NEWS_DIR, f'{filename}_{counter}.txt')
            counter += 1

        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(f"TIÊU ĐỀ: {result['title']}\n")
            f.write(f"URL: {full_url}\n")
            f.write(f"TÁC GIẢ: {result['author']}\n")
            f.write(f"\n{'='*80}\n\n")
            f.write(result['content'])

        result['saved_file'] = txt_path
        print(f'      [OK] Saved: {filename[:40]}.txt')

    except Exception as e:
        result['status'] = 'failed'
        result['error'] = str(e)
        print(f'      [FAIL] {str(e)[:50]}')

    return result

async def crawl_news_list(cfg, options):
    url = cfg.get('url')
    max_pages = options.get('max_pages', 100)
    max_articles = options.get('max_files', 0)
    delay = options.get('delay_between_requests', 3)

    result = {'url': url, 'articles': [], 'status': 'pending'}

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=[
                '--disable-blink-features=AutomationControlled',
                '--disable-features=IsolateOrigins,site-per-process',
                '--no-sandbox',
                '--disable-setuid-sandbox',
                '--disable-dev-shm-usage',
                '--disable-accelerated-2d-canvas',
                '--no-first-run',
                '--no-zygote',
                '--disable-gpu'
            ]
        )

        context = await browser.new_context(
            viewport={'width': 1920, 'height': 1080},
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
            locale='vi-VN',
            timezone_id='Asia/Ho_Chi_Minh',
        )

        page = await context.new_page()

        # Apply stealth
        await stealth_async(page)

        try:
            print(f'\n  Accessing: {url}')

            # Thử truy cập trang chủ trước
            print('  Step 1: Visit homepage first...')
            await page.goto('https://www.mod.gov.vn/', wait_until='domcontentloaded', timeout=60000)
            await page.wait_for_timeout(3000)

            # Sau đó mới vào trang news
            print('  Step 2: Navigate to news page...')
            await page.goto(url, wait_until='domcontentloaded', timeout=60000)
            await page.wait_for_timeout(5000)

            html_check = await page.content()

            # Debug: Save HTML
            with open('test_page.html', 'w', encoding='utf-8') as f:
                f.write(html_check)
            print('  Saved test_page.html for inspection')

            if '403 Forbidden' in html_check or 'Access Denied' in html_check:
                print('  ERROR: Still getting 403 Forbidden!')
                print('  This website has strong anti-bot protection.')
                print('  You may need to:')
                print('    1. Use residential proxy')
                print('    2. Try from different IP')
                print('    3. Use browser automation tools like Selenium with real browser profile')
                result['status'] = 'failed'
                result['error'] = '403 Forbidden - Anti-bot protection'
                await browser.close()
                return result

            print('  ✓ Page loaded successfully!')

            # Scroll
            for _ in range(3):
                await page.evaluate('window.scrollTo(0, document.body.scrollHeight)')
                await page.wait_for_timeout(2000)
            await page.evaluate('window.scrollTo(0, 0)')
            await page.wait_for_timeout(2000)

            current_page = 1

            while current_page <= max_pages:
                print(f'\n  === Page {current_page} ===')
                await page.wait_for_timeout(delay * 1000)

                html = await page.content()
                soup = BeautifulSoup(html, 'lxml')

                article_items = soup.select('div.itemnew-inlist')
                item_count = len(article_items)
                print(f'    Found {item_count} articles')

                if item_count == 0:
                    print('    No articles found')
                    break

                for i, item in enumerate(article_items):
                    if max_articles > 0 and len(result['articles']) >= max_articles:
                        print(f'    Reached limit ({max_articles})')
                        break

                    try:
                        link_tag = item.select_one('h5.titleLeftNews a')
                        if link_tag and link_tag.get('href'):
                            article_url = link_tag['href']
                            article_title = link_tag.get_text(strip=True)

                            time_tag = item.select_one('div.time-post')
                            article_time = time_tag.get_text(strip=True) if time_tag else ''

                            print(f'    [{i+1}] {article_title[:50]}...')

                            article_data = await crawl_news_article(page, article_url, url)
                            article_data['time_post'] = article_time
                            article_data['title_from_list'] = article_title

                            result['articles'].append(article_data)

                            await page.goto(page.url, wait_until='domcontentloaded')
                            await page.wait_for_timeout(delay * 1000)

                    except Exception as e:
                        print(f'    Error: {str(e)[:50]}')

                if max_articles > 0 and len(result['articles']) >= max_articles:
                    break

                try:
                    next_page_num = current_page + 1
                    next_link = page.locator(f'div.page a[title*="page {next_page_num}"]')

                    if await next_link.count() > 0:
                        print(f'    Going to page {next_page_num}...')
                        await next_link.first.click()
                        await page.wait_for_timeout(5000)
                        current_page += 1
                    else:
                        break
                except Exception as e:
                    print(f'    Pagination error: {str(e)[:50]}')
                    break

            result['status'] = 'success'
            result['total_pages'] = current_page

        except Exception as e:
            result['status'] = 'failed'
            result['error'] = str(e)
        finally:
            await browser.close()

    return result

async def main():
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)

    urls = data.get('urls', [])
    opts = data.get('options', {})

    print(f'URLs: {len(urls)}')
    print('='*50)

    results = []
    for i, cfg in enumerate(urls, 1):
        url = cfg.get('url')
        print(f'\n[{i}] {url}')

        r = await crawl_news_list(cfg, opts)

        success = sum(1 for a in r['articles'] if a['status'] == 'success')
        print(f'\n  Crawled: {success}/{len(r["articles"])}')

        results.append(r)

    return results

all_results = asyncio.get_event_loop().run_until_complete(main())

output = {
    'results': all_results,
    'summary': {
        'urls': len(all_results),
        'total_articles': sum(len(r['articles']) for r in all_results),
        'successful': sum(sum(1 for a in r['articles'] if a['status']=='success') for r in all_results)
    },
    'crawled_at': datetime.now().isoformat()
}

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f'\n{"="*50}')
print(f'Total: {output["summary"]["total_articles"]}')
print(f'Successful: {output["summary"]["successful"]}')
print(f'Saved: {OUTPUT_FILE}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Installing dependencies...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 3,917 B in 1s (2,778 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry mi